# Phase 4: Training Pipeline (tabular_lgbm)

In [1]:
# Cell 1: Environment Setup & Hardware Detection
import os
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

def find_project_root(indicator_file="configs/datasets.yaml"):
    # Dynamically find the project root relative to the notebook location.
    curr = os.getcwd()
    # Check current and up to 4 parent levels (Notebooks are usually in notebooks/)
    for _ in range(5):
        if os.path.exists(os.path.join(curr, indicator_file)):
            return curr
        curr = os.path.dirname(curr)
    # Fallback for local dev or standard Colab location
    return '/content/drive/MyDrive/seismic-first-break-picking'

PROJECT_ROOT = find_project_root()

if IN_COLAB:
    # Ensure all required packages are installed in the fresh Colab runtime
    !pip install -q mlflow optuna lightgbm segmentation-models-pytorch pyyaml

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

import torch
import numpy as np
import random
import matplotlib.pyplot as plt

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device_type = 'cuda' if torch.cuda.is_available() else 'cpu'
device = torch.device(device_type)
device_name = torch.cuda.get_device_name(0) if device_type == 'cuda' else 'cpu'
vram_bytes = torch.cuda.get_device_properties(0).total_memory if device_type == 'cuda' else 0

print(f"Device: {device_name}")
print(f"VRAM: {vram_bytes / 1e9:.2f} GB" if vram_bytes > 0 else "")


Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 71.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━

In [2]:
# Cell 2: Config & MLflow Setup
import mlflow
import types
from src.utils.config_loader import load_model_config
from src.training.mlflow_logger import MLFlowLogger

config_path = os.path.join(PROJECT_ROOT, 'configs', 'model_tabular_lgbm.yaml')
config = load_model_config(config_path, as_namespace=True)

# Defensive fallback: guarantee config.output exists with all required keys
if not hasattr(config, 'output'):
    config.output = types.SimpleNamespace()
if not hasattr(config.output, 'tracking_uri'):
    config.output.tracking_uri = f'file:///{PROJECT_ROOT}/mlruns'
if not hasattr(config.output, 'checkpoint_dir'):
    config.output.checkpoint_dir = f'models/tabular_lgbm'

# CRITICAL: Resolve checkpoint_dir to an absolute path inside Google Drive.
# If it is a relative path (e.g. 'models/cnn1d'), Colab resolves it against
# CWD=/content (ephemeral disk) ï¿½ models would be LOST when the session ends.
# Prepend PROJECT_ROOT so all checkpoints land on Drive unconditionally.
if not os.path.isabs(config.output.checkpoint_dir):
    config.output.checkpoint_dir = os.path.join(PROJECT_ROOT, config.output.checkpoint_dir)
os.makedirs(config.output.checkpoint_dir, exist_ok=True)
print(f"Checkpoint dir (absolute): {config.output.checkpoint_dir}")

# Terminate any active MLflow run from a previous cell execution (rerun safety)
mlflow.end_run()

# --- Resume-safe MLflow init ---
# _save_checkpoint writes mlflow_run_id.txt on every epoch save.
# If the file exists, we rejoin that run so all epochs appear on one timeline.
# If the run was FINISHED (or the file is stale) resume_run() transparently
# falls back to a fresh run, so re-running a completed notebook is fully safe.
_run_id_file = os.path.join(config.output.checkpoint_dir, 'mlflow_run_id.txt')
_existing_run_id = None
if os.path.exists(_run_id_file):
    with open(_run_id_file) as _f:
        _existing_run_id = _f.read().strip() or None

logger = MLFlowLogger(config.output.tracking_uri, config.experiment.name)
if _existing_run_id:
    logger.resume_run(_existing_run_id)
    print(f"Resumed MLflow run: {_existing_run_id}")
else:
    logger.start_run()

logger.log_params(config)
print(f"Loaded config for: {config.experiment.name}")
print(f"Checkpoint dir: {config.output.checkpoint_dir}")


Checkpoint dir (absolute): /content/drive/MyDrive/seismic-first-break-picking/models/tabular_lgbm


/usr/local/lib/python3.12/dist-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/04/13 07:30:41 INFO mlflow.tracking.fluent: Experiment with name 'tabular_lgbm' does not exist. Creating a new experiment.


Loaded config for: tabular_lgbm
Checkpoint dir: /content/drive/MyDrive/seismic-first-break-picking/models/tabular_lgbm


In [3]:
# Cell 3: Checkpoint State Detection
import os
import torch

# Trainer saves as '{experiment_name}_latest.pt' ï¿½ match that filename exactly.
checkpoint_path = os.path.join(config.output.checkpoint_dir, f'{config.experiment.name}_latest.pt')
is_progressive = hasattr(config, 'progressive_training') and getattr(config.training, 'training_mode', 'combined') == 'progressive'

if not os.path.exists(checkpoint_path):
    state = 'NO_CHECKPOINT_EXISTS'
    start_asset_index = 0
    resume_epoch = 0
    completed_assets = []
else:
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    training_state = checkpoint.get('training_state', {})
    resume_epoch = checkpoint.get('epoch', 0)

    if is_progressive:
        if training_state.get('is_fully_trained', False):
            state = 'TRAINING_COMPLETE'
            completed_assets = config.progressive_training.asset_order
            start_asset_index = len(completed_assets)
        else:
            completed_assets = training_state.get('completed_assets', [])
            start_asset_index = len(completed_assets)
            state = f"ASSET_{start_asset_index}_COMPLETE" if completed_assets else "NO_CHECKPOINT_EXISTS"
    else:
        # Combined Training (Default)
        completed_assets = []
        start_asset_index = 0
        state = 'RESUMING_COMBINED_TRAINING'

print(f"Detected state: {state}")
print(f"Resuming from epoch: {resume_epoch}")


Detected state: NO_CHECKPOINT_EXISTS
Resuming from epoch: 0


In [4]:
# Cell 4: Data & Sampler Setup
from torch.utils.data import DataLoader
from src.data.dataset import ShotGatherDataset, ProgressiveAssetSampler, variable_width_collate_fn, trace_collate_fn
from src.data.transforms import build_transforms
from src.utils.config_loader import load_yaml
csv_path = config.data.index_csv if os.path.isabs(config.data.index_csv) else os.path.join(PROJECT_ROOT, config.data.index_csv)
# Load augmentation config and build training transforms
preproc_path = os.path.join(PROJECT_ROOT, 'configs', 'preprocessing.yaml')
preproc_cfg = load_yaml(preproc_path)
train_transform = build_transforms(preproc_cfg.get('augmentation', {}), is_training=True)
print(f"Train augmentations: {train_transform}")
train_ds = ShotGatherDataset(csv_path, split='train', transform=train_transform)
val_ds = ShotGatherDataset(csv_path, split='val')  # No augmentation for validation
val_loader = DataLoader(
    val_ds, batch_size=config.training.batch_size, shuffle=False,
    collate_fn=trace_collate_fn, num_workers=0, pin_memory=True
)
train_loader = DataLoader(
    train_ds, batch_size=config.training.batch_size, shuffle=True,
    collate_fn=trace_collate_fn, num_workers=0, pin_memory=True
)
print(f"Train samples: {len(train_ds)} | Val samples: {len(val_ds)}")


Train augmentations: Compose([
])
Train samples: 2907 | Val samples: 621


In [5]:
# Cell 5: Model Map
from src.models.tabular import LightGBMWrapper
model = LightGBMWrapper()
print("LightGBM Ready")

LightGBM Ready


In [ ]:
# Cell 7: LightGBM Fitting on Train Set
import numpy as np
from src.features.features import extract_features

print("Fitting LightGBM on train_loader features...")
all_features, all_labels = [], []

max_batches = 100  # Cap at ~25,000 traces to prevent RAM exhaustion and overhead
for i, batch in enumerate(train_loader):
    if i >= max_batches:
        print(f"Early breaking LightGBM feature extraction safely at {max_batches} batches.")
        break
    traces = batch['traces'].squeeze(1).numpy()
    labels = batch['labels_ms'].numpy()
    mask = batch['valid_mask'].numpy()  # always present: trace_collate emits it
    offsets = batch.get('offsets', np.zeros(traces.shape[0]))
    if mask.any():
        f = extract_features(traces, offsets)
        valid_rows = mask.astype(bool) if mask.ndim == 1 else mask.any(axis=1)
        all_features.append(f[valid_rows])
        all_labels.extend(labels[valid_rows].tolist())

if all_features:
    X = np.vstack(all_features)
    y = np.array(all_labels)
    print(f"Fitting on {{X.shape[0]}} traces with {{X.shape[1]}} features...")
    model.model.fit(X, y)
    print("LightGBM fitting complete!")

    # Persist to Google Drive (survives session resets)
    import joblib
    lgbm_save_path = os.path.join(config.output.checkpoint_dir, 'lgbm_model.pkl')
    os.makedirs(config.output.checkpoint_dir, exist_ok=True)
    joblib.dump(model.model, lgbm_save_path)
    print(f"Model saved to Drive: {{lgbm_save_path}}")
    logger.log_artifact(lgbm_save_path)
else:
    print("WARNING: No valid training traces found.")


Fitting LightGBM on train_loader features...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [ ]:
# Cell 8: Universal Evaluation (all model tiers)
# Force-purge ALL src.* modules from Colab's module cache so that any fixes
# pushed to Drive (loss.py, trainer.py, dataset.py, evaluator.py, etc.) are
# reloaded fresh. Without this, Colab reuses stale bytecode from the session's
# first import, silently ignoring edits on Drive.
import importlib, sys
_stale = [k for k in sys.modules if k.startswith('src.')]
for _mod_name in _stale:
    del sys.modules[_mod_name]
print(f"Cache cleared: {len(_stale)} src.* modules evicted.")

from src.training.evaluator import ModelEvaluator

_is_dl = 'tabular_lgbm' not in ('classical', 'tabular_lgbm')
_history = trainer.history if 'trainer' in dir() and hasattr(trainer, 'history') else {}

evaluator = ModelEvaluator(
    model=model,
    val_loader=val_loader,
    logger=logger,
    device=device,
    model_key='tabular_lgbm',
    is_dl=_is_dl,
    history=_history,
)
final_metrics = evaluator.run()

# Close MLflow run cleanly
import mlflow
mlflow.end_run()
print("MLflow run closed.")
